[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/notebooks/EN/chapter0_seminar_notebook.ipynb)

# Seminar 0: Financial Markets in 2026 — Hands-on

**Modelling Financial Markets (MFM)**, Master's programme *Applied Statistics and Data Science*, Bucharest University of Economic Studies

*Daniel Traian Pele, Antoaneta Amza*

Exercises of Seminar 0.

- **Part A**: pen-and-paper warm-up (A1–A4) and derivations (A5–A9).
- **Part B**: verification and inference on real data (B1–B8); every exercise ends with interpretation questions.
- **Part C**: an open question, a starting point for your team project.

**Solved** problems (tag [Solved]) show the full code and results: use them as models. **Proposed** problems (tag [Proposed]) have an empty cell: write your code there. The data loader and helper functions are provided.

**In class:** A1–A4, B6, B1, B3, B1 extended, C2; **at home:** B2; all other sections are optional extension packs (see the seminar slides).

## Setup

In [ ]:
import os
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# MFM chart style: transparent background, English labels, legend below
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Brand colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'


def save_fig(name):
    """Show the chart inline"""
    plt.show()
    plt.close()


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def bottom_legend(fig, ncol=3, handles=None, labels=None, fontsize=8):
    """Legenda sub figura (in afara axelor), dupa tight_layout; save_fig o include (bbox_inches='tight')."""
    plt.tight_layout()
    if handles is None:
        handles, labels, seen = [], [], set()
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in seen and not l.startswith('_'):
                    handles.append(h); labels.append(l); seen.add(l)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=ncol, frameon=False, fontsize=fontsize)


def log_axis(ax, ticks):
    """Axa logaritmica cu etichete lizibile (fara notatie stiintifica)."""
    ax.set_yscale('log')
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
LOCAL_DIRS = ['../../data/market', '../data/market', 'data/market']   # cloned repository
START, END = '2000-01-01', '2026-09-18'

# name -> (source, symbol, field)
SERIES = {
    'S&P 500': ('market', 'GSPC.INDX', 'close'),
    'Euro Stoxx 50': ('market', 'STOXX50E.INDX', 'close'),
    'Nikkei 225': ('market', 'N225.INDX', 'close'),
    'VIX': ('market', 'VIX.INDX', 'close'),
    'BET-TR': ('market', 'BETTR.INDX', 'close'),
    'BET': ('market', 'BET', 'close'),
    'BET-XT': ('market', 'BETXT.INDX', 'close'),
    'US Treasuries 20y+ (TLT)': ('market', 'TLT.US', 'adjusted_close'),
    'SPY': ('market', 'SPY.US', 'adjusted_close'),
    'RSP': ('market', 'RSP.US', 'adjusted_close'),
    'IBIT': ('market', 'IBIT.US', 'close'),
    'IBIT volume': ('market', 'IBIT.US', 'volume'),
    'Gold': ('market', 'XAUUSD.FOREX', 'close'),
    'EUR/USD': ('market', 'EURUSD.FOREX', 'close'),
    'EUR/RON (EODHD)': ('market', 'EURRON.FOREX', 'close'),
    'Bitcoin': ('market', 'BTC-USD.CC', 'close'),
    'Ethereum': ('market', 'ETH-USD.CC', 'close'),
    'Tether (USDT)': ('market', 'USDT-USD.CC', 'close'),
    'Romania 10y': ('market', 'RO10Y.GBOND', 'close'),
    'Germany 10y': ('market', 'DE10Y.GBOND', 'close'),
    'Banca Transilvania': ('market', 'TLV.RO', 'adjusted_close'),
    'OMV Petrom': ('market', 'SNP.RO', 'adjusted_close'),
    'BRD': ('market', 'BRD.RO', 'adjusted_close'),
    'Transgaz': ('market', 'TGN.RO', 'adjusted_close'),
    'Romgaz': ('market', 'SNG.RO', 'adjusted_close'),
    'Hidroelectrica': ('market', 'H2O.RO', 'adjusted_close'),
    'EUR/RON': ('bnr', 'EUR', None),
    'DGS10': ('fred', 'DGS10', None),
    'DGS2': ('fred', 'DGS2', None),
    'FEDFUNDS': ('fred', 'FEDFUNDS', None),
    'USD per EUR': ('fred', 'DEXUSEU', None),
    'JPY per USD': ('fred', 'DEXJPUS', None),
    'T10Y2Y': ('fred', 'T10Y2Y', None),
    'ETF equities': ('fred', 'BOGZ1LM563064100Q', None),
    'All equities': ('fred', 'BOGZ1LM893064105Q', None),
    'Stablecoins': ('defillama', 'stablecoincharts/all', None),
}


def read_market(symbol):
    """Daily EODHD series: local copy if the repository is cloned, otherwise GitHub"""
    for d in LOCAL_DIRS:
        path = os.path.join(d, f'{symbol}.csv')
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True)
    return pd.read_csv(REPO_RAW + f'{symbol}.csv', index_col='date', parse_dates=True)


def fetch_daily(symbol, source='market', field='close', start=START, end=END):
    """Punctul unic de acces la date: intoarce o pd.Series zilnica (fara NaN)."""
    if source == 'market':
        s = read_market(symbol)[field]
    elif source == 'fred':
        raw = urllib.request.urlopen(
            f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={symbol}', timeout=90).read().decode()
        s = pd.read_csv(io.StringIO(raw), index_col=0, parse_dates=True, na_values='.').iloc[:, 0]
    elif source == 'defillama':
        req = urllib.request.Request(f'https://stablecoins.llama.fi/{symbol}',
                                     headers={'User-Agent': 'Mozilla/5.0'})
        d = json.load(urllib.request.urlopen(req, timeout=90))
        s = pd.Series({pd.to_datetime(int(x['date']), unit='s'): x['totalCirculatingUSD'].get('peggedUSD', 0)
                       for x in d}) / 1e9                                   # miliarde USD
    elif source == 'bnr':
        # cursul oficial BNR (RON pentru o unitate de valuta), arhive XML anuale
        import re
        rows = []
        for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
            url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
            xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                         timeout=90).read().decode()
            for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
                m = re.search(rf'<Rate currency="{symbol}">([\d.]+)</Rate>', body)
                if m:
                    rows.append((d, float(m.group(1))))
        s = pd.Series(dict(rows))
    else:
        raise ValueError(f'unknown source: {source}')
    s = pd.to_numeric(s, errors='coerce')
    s.index = pd.to_datetime(s.index)
    s.index.name = 'Date'
    return s.sort_index().loc[start:end].dropna()


WEEKDAYS_ONLY = {'EUR/USD', 'Gold'}   # FX/OTC quotes: weekend prints break the Friday-Monday return


def load(name, start=START, end=END):
    """Seria cu numele din SERIES (pentru aur si EUR/USD: doar zilele lucratoare)."""
    source, symbol, field = SERIES[name]
    s = fetch_daily(symbol, source, field, start, end).rename(name)
    if name in WEEKDAYS_ONLY:
        s = s[s.index.dayofweek < 5]
    return s


def load_panel(names, start=START, end=END):
    """Mai multe serii aliniate pe data (NaN acolo unde o piata nu tranzactioneaza)."""
    return pd.concat([load(n, start, end) for n in names], axis=1)

In [ ]:
def clean_series(y, max_dev=1.0, window=11):
    """Elimina weekend-urile si punctele aflate la peste max_dev de mediana mobila centrata."""
    y = y[y.index.dayofweek < 5]
    med = y.rolling(window, center=True, min_periods=3).median()
    return y[(y - med).abs() <= max_dev]


def drawdown(p):
    """Drawdown fata de maximul anterior: P_t / max_{s<=t} P_s - 1."""
    return p / p.cummax() - 1


def ann_stats(p, periods=252):
    """Randament anualizat (CAGR) si volatilitate anualizata din preturi zilnice.

    periods='obs' foloseste numarul efectiv de observatii pe an al seriei.
    """
    p = p.dropna()
    years = (p.index[-1] - p.index[0]).days / 365.25
    cagr = (p.iloc[-1] / p.iloc[0]) ** (1 / years) - 1
    r = np.log(p).diff().dropna()
    if periods == 'obs':
        periods = len(r) / years
    return cagr, r.std() * np.sqrt(periods)


def to_usd(local, fx, invert=False):
    """Converteste un indice in moneda locala in USD cu cursul FRED din aceeasi zi
    (ultimul curs disponibil, cel mult 5 zile in urma, daca ziua lipseste)."""
    f = fx.dropna().reindex(local.index, method='ffill', tolerance=pd.Timedelta(days=5))
    return (local / f if invert else local * f).dropna()

In [ ]:
def sb_index(n, mean_block, rng):
    """Stationary bootstrap (Politis and Romano, 1994): blocks of random, geometric length with mean mean_block, wrapped around"""
    idx = np.empty(n, dtype=int); i = rng.integers(n)
    for t in range(n):
        i = rng.integers(n) if (t == 0 or rng.random() < 1 / mean_block) else (i + 1) % n
        idx[t] = i
    return idx


def obs_per_year(p):
    """Observed trading days per year (about 252 for equities, 365 for crypto, about 260 for spot gold on weekdays)"""
    return (len(p) - 1) / ((p.index[-1] - p.index[0]).days / 365.25)   # return intervals per year


def sharpe(R, q):
    """Annualised Sharpe ratio, r_f = 0: mean / sd of simple returns R times sqrt(q)"""
    return R.mean() / R.std(ddof=1) * np.sqrt(q)


def se_lo(R, q):
    """Lo (2002), iid: SE of the annualised Sharpe from T daily returns, sqrt(q (1 + SR_d^2/2) / T), about 1/sqrt(Y)"""
    sr_d = R.mean() / R.std(ddof=1)
    return np.sqrt(q * (1 + sr_d ** 2 / 2) / len(R))


def se_opdyke(R, q):
    """Opdyke (2007): SE of the annualised Sharpe corrected for skewness and kurtosis of daily returns"""
    from scipy import stats
    sr = R.mean() / R.std(ddof=1); g3 = stats.skew(R); g4 = stats.kurtosis(R, fisher=False)
    return np.sqrt((1 + sr ** 2 / 2 - g3 * sr + (g4 - 3) / 4 * sr ** 2) / len(R)) * np.sqrt(q)


def drawdown_spells(p):
    """Spells below a previous peak; convention: a spell starts ON the peak date and ends on the first close >= peak"""
    peak = p.cummax(); spells, start = [], None
    for t, below in (p < peak).items():
        if below and start is None:
            start = p.loc[:t][p.loc[:t] == peak.loc[t]].index[-1]
        if not below and start is not None:
            spells.append((start, t)); start = None
    if start is not None:
        spells.append((start, None))
    return spells


def hampel_flags(x, c, floor=0.0, window=11):
    """Hampel filter (Hampel, 1974): flag |x_t - m_t| > max(c * 1.4826 * MAD_t, floor) over a centred window"""
    v, h = x.values, window // 2; med, mad = np.full(len(v), np.nan), np.full(len(v), np.nan)
    for t in range(len(v)):
        w = v[max(0, t - h):t + h + 1]                    # window truncated at the ends of the sample
        med[t] = np.median(w); mad[t] = np.median(np.abs(w - med[t]))   # MAD_t = median_j |x_j - m_t|
    return pd.Series(np.abs(v - med) > np.maximum(c * 1.4826 * mad, floor), index=x.index)


def fisher_ci(r, n, z=1.96):
    """95% CI for a correlation with Fisher's z transform (iid pairs from a bivariate Normal distribution)"""
    se = 1 / np.sqrt(n - 3)
    return np.tanh(np.arctanh(r) - z * se), np.tanh(np.arctanh(r) + z * se)


rng = np.random.default_rng(42)

## Part A: pen-and-paper exercises

**A1–A4 [Solved] (warm-up).** A1: path 100 → 110 → 99, simple vs log returns. A2: drawdown of 100, 120, 90, 110, 130, 100. A3: annualise a daily volatility of 1% for an equity index and for Bitcoin. A4: CAGR of a ×3.72 multiple over 11.71 years.

In [ ]:
P = np.array([100, 110, 99]); R = P[1:] / P[:-1] - 1; r = np.log(P[1:] / P[:-1])
print('A1: R =', R.round(4), 'sum', round(float(R.sum()), 4), '| r =', r.round(4), 'sum', round(float(r.sum()), 5), '= ln(0.99)', round(float(np.log(0.99)), 5))
P = pd.Series([100, 120, 90, 110, 130, 100]); print('A2: DD =', drawdown(P).round(4).tolist(), 'max', round(float(drawdown(P).min()), 4))
print('A3:', round(0.01 * np.sqrt(252), 4), round(0.01 * np.sqrt(365), 4))
print('A4: CAGR =', round(3.72 ** (1 / 11.71) - 1, 4))

**A5 [Solved]. Volatility drag.** (a) Expand $\ln(1+R)$ to second order and show $\mu_{geo} \approx \mu_{arit} - \sigma^2/2$. (b) Bitcoin (Chapter 1): daily log mean and standard deviation; annualised log mean, $\sigma^2/2$, implied simple mean and CAGR. (c) An asset has an annual arithmetic mean of 10% and a volatility of 60%: which CAGR do you expect?

In [ ]:
btc = load('Bitcoin', start='2014-09-17')
r = np.log(btc).diff().dropna(); R = np.expm1(r)
m_d, s_d = r.mean(), r.std()
mu_log = m_d * 365; half_var = s_d ** 2 * 365 / 2
years = (btc.index[-1] - btc.index[0]).days / 365.25
print(f'(b) daily log mean {m_d:.4%}, daily sd {s_d:.3%}; annual log mean {mu_log:.1%}, sigma^2/2 {half_var:.3f}')
print(f'    implied simple mean {mu_log + half_var:.1%} (direct: {R.mean() * 365:.1%}); CAGR = e^mu - 1 = {np.exp(mu_log) - 1:.1%} '
      f'(from prices {(btc.iloc[-1] / btc.iloc[0]) ** (1 / years) - 1:.1%})')
print(f'(c) log growth 0.10 - 0.60^2/2 = {0.10 - 0.6 ** 2 / 2:.2f}, CAGR = {np.exp(0.10 - 0.18) - 1:.1%}')

**A6 [Proposed]. When the $\sqrt{P}$ rule fails.** (a) Show $\operatorname{Var}(\sum_{t=1}^{k} r_t) = k\sigma^2[1 + 2\sum_{j=1}^{k-1}(1-j/k)\rho_j]$. (b) AR(1), $\rho_j = \rho^j$: find the limit of the ratio to $k\sigma^2$. (c) EUR/RON (BNR): correct the $\sqrt{252}$ volatility assuming AR(1). (d) The EODHD series with bad ticks has a negative $\hat\rho_1$: in which direction is $\sqrt{252}$ wrong now? (e) Check the AR(1) correction in (c) with monthly returns.

In [ ]:
# Your code here

**A7 [Proposed]. Why is the weekly correlation lower than the daily one?** (a) For 5-day sums $X, Y$ of daily returns without own autocorrelation, show $\operatorname{Cov}(X,Y) = 5\gamma(0) + \sum_{j=1}^{4}(5-j)[\gamma(j) + \gamma(-j)]$, with $\gamma(j) = \operatorname{Cov}(x_t, y_{t-j})$. (b) B3: 0.42 daily vs 0.30 weekly (2022–2026): which sign must the lagged covariances have? Measure them. (c) With about 246 weeks, is the difference clearly significant? (d) Recompute the weekly correlation with other anchor days.

In [ ]:
# Your code here

**A8 [Proposed]. The standard error of a Sharpe ratio.** $r_t$ iid with mean $\mu$, variance $\sigma^2$, skewness $\gamma_3$, kurtosis $\gamma_4$. (a) Use the delta method on $\sqrt{T}(\hat\mu - \mu, \hat\sigma^2 - \sigma^2)$ to get the asymptotic variance of $\sqrt{T}(\widehat{SR} - SR)$. (b) Show it equals $1 + SR^2/2$ for Normal returns (Lo, 2002). (c) Annualise, $SR_a = \sqrt{q}\,SR_d$, $T = qY$: show $\operatorname{Var}(\widehat{SR}_a) \approx 1/Y$ whatever $q$. (d) Compute the SE for SPY with (b) and with (a); compare with B1 extended.

In [ ]:
# Your code here

**A9 [Proposed]. Asynchronous closes.** Log prices $X = W + U$, $Y = W + V$ ($W, U, V$ independent Brownian motions); $X$ closes at $t$, $Y$ at $t - \delta$. (a) Show $\operatorname{Corr}(x_t, y_t) = \rho(1-\delta)$ and $\operatorname{Corr}(x_t, y_{t+1}) = \rho\delta$. (b) In this model the sum of the lag $-1, 0, +1$ correlations recovers $\rho$ (Scholes and Williams, 1977). (c) Bitcoin (price of day $t$ at 24:00 UTC) vs S&P 500 (20:00 or 21:00 UTC): $\delta$ and the implied attenuation. (d) Measure the lead/lag correlations in 2022–2026.

In [ ]:
# Your code here

## Part B: verification and inference

### B1 [Solved]. Cross-asset dashboard

**Task.** All series in USD since 2 Jan 2015 (S&P 500 with dividends = SPY; Euro Stoxx 50 and Nikkei 225 converted with FRED rates; gold; TLT; Bitcoin); growth of 1 USD on a log scale; CAGR and annualised volatility, annualising with the observed number of trading days per year.

**Interpretation.** Gold: weekend prints are removed (weekday quotes only, about 260 a year). How much does annualising with 252 instead of the actual frequency change its volatility? Check against the GLD ETF.

In [ ]:
px = load_panel(['SPY', 'Euro Stoxx 50', 'Nikkei 225', 'Gold', 'US Treasuries 20y+ (TLT)', 'Bitcoin'])
fx_usd = load_panel(['USD per EUR', 'JPY per USD'])
assets = {'S&P 500 TR (SPY)': px['SPY'].dropna(),
          'Euro Stoxx 50 (USD)': to_usd(px['Euro Stoxx 50'].dropna(), fx_usd['USD per EUR']),
          'Nikkei 225 (USD)': to_usd(px['Nikkei 225'].dropna(), fx_usd['JPY per USD'], invert=True),
          'Gold': px['Gold'].dropna(), 'TLT': px['US Treasuries 20y+ (TLT)'].dropna(), 'Bitcoin': px['Bitcoin'].dropna()}
assets = {k: v.loc['2015-01-02':] for k, v in assets.items()}
fig, ax = plt.subplots(figsize=(7, 3)); rows = []
for n, p in assets.items():
    ax.plot(p.index, p / p.iloc[0], lw=0.8, label=n)
    q = obs_per_year(p); cagr, _ = ann_stats(p); vol = np.log(p).diff().dropna().std() * np.sqrt(q)
    rows.append((n, round(q), p.iloc[-1] / p.iloc[0], cagr, vol, np.log(p).diff().dropna().std() * np.sqrt(252)))
log_axis(ax, [0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500]); legend_outside_bottom(ax, ncol=3, y=-0.12); plt.show()
gld = fetch_daily('GLD.US', 'market', 'adjusted_close', start='2015-01-02'); print(f"GLD cross-check: vol {np.log(gld).diff().dropna().std() * np.sqrt(252):.3f} (252 trading days)")
pd.DataFrame(rows, columns=['asset', 'days/yr', 'multiple', 'CAGR', 'vol', 'vol with 252']).set_index('asset').round(3)

### B1 extended [Solved]. How reliable are the numbers in the table?

**Task.** For each asset: Sharpe ratio ($r_f = 0$) and its standard error (i) iid Normal returns, Lo (2002), from the daily $SR_d$ of $T$ daily returns: $\sqrt{q(1+SR_d^2/2)/T} \approx 1/\sqrt{Y}$; (ii) corrected for skewness and kurtosis, Opdyke (2007), with the daily SR; (iii) stationary bootstrap (Politis and Romano, 1994), mean block 20 days, 2,000 replications: 95% intervals for CAGR and Sharpe; sensitivity to blocks of 5 and 60 days; (iv) 95% interval for the Sharpe difference S&P 500 minus gold on common days; (v) the same for every pair of the five non-crypto assets, with a Bonferroni correction for the 10 comparisons.

**Interpretation.** Which differences between assets survive once uncertainty is added? Why is the iid formula probably too optimistic for Bitcoin? What does the dependence of the intervals on the block length tell you?

In [ ]:
rows = []
for n, p in assets.items():
    q = obs_per_year(p); years = (p.index[-1] - p.index[0]).days / 365.25
    r = np.log(p).diff().dropna().values; R = np.expm1(r); sr = sharpe(R, q)
    row = [n, sr, se_lo(R, q), se_opdyke(R, q)]
    for mb in (5, 20, 60):
        cs, ss = [], []
        for _ in range(2000 if mb == 20 else 1000):
            i = sb_index(len(r), mb, rng); cs.append(np.exp(r[i].sum() / years) - 1); ss.append(sharpe(R[i], q))
        row += [f'[{np.percentile(ss, 2.5):.2f}, {np.percentile(ss, 97.5):.2f}]']
        if mb == 20:
            row += [f'[{np.percentile(cs, 2.5):.3f}, {np.percentile(cs, 97.5):.3f}]']
    rows.append(row)
cols = ['asset', 'Sharpe', 'SE Lo', 'SE Opdyke', 'SR CI b=5', 'SR CI b=20', 'CAGR CI b=20', 'SR CI b=60']
print(pd.DataFrame(rows, columns=cols).set_index('asset').round(3).to_string())
j = np.log(pd.concat([assets['S&P 500 TR (SPY)'], assets['Gold']], axis=1).dropna()).diff().dropna(); Rj = np.expm1(j.values)
d0 = sharpe(Rj[:, 0], 252) - sharpe(Rj[:, 1], 252)
ds = [sharpe(Rj[i, 0], 252) - sharpe(Rj[i, 1], 252) for i in (sb_index(len(Rj), 20, rng) for _ in range(2000))]
print(f'Sharpe S&P 500 - gold (common days, n = {len(Rj)}): {d0:+.2f}, 95% CI [{np.percentile(ds, 2.5):+.2f}, {np.percentile(ds, 97.5):+.2f}]')
# (v) every pair of the five non-crypto assets: joint bootstrap of the difference on common days; Bonferroni: 10 tests, each at 0.5%
import itertools
rng_pairs = np.random.default_rng(2026)
for a, b in itertools.combinations([k for k in assets if k != 'Bitcoin'], 2):
    j = np.log(pd.concat([assets[a], assets[b]], axis=1).dropna()).diff().dropna(); Rj = np.expm1(j.values)
    ds = np.array([sharpe(Rj[i, 0], 252) - sharpe(Rj[i, 1], 252) for i in (sb_index(len(Rj), 20, rng_pairs) for _ in range(2000))])
    p = min(1, 2 * min((ds <= 0).mean(), (ds >= 0).mean()))
    print(f'{a} - {b}: {sharpe(Rj[:, 0], 252) - sharpe(Rj[:, 1], 252):+.2f}, 95% CI [{np.percentile(ds, 2.5):+.2f}, {np.percentile(ds, 97.5):+.2f}], '
          f'99.5% CI [{np.percentile(ds, 0.25):+.2f}, {np.percentile(ds, 99.75):+.2f}], bootstrap p = {p:.3f}')

### B2 [Proposed]. Drawdowns and recovery time

**Task.** Drawdown series of the S&P 500 (2000–2026) and Bitcoin (Sep 2014–2026): maximum drawdown and its date; longest spell below a previous peak. **Convention:** a spell starts on the peak date and ends on the first close at or above the peak.

**Interpretation.** Bitcoin's peak date depends on the data source's daily close convention (24:00 UTC snapshot, the end of the UTC day, or another time zone). Which of your results are sensitive to it?

In [ ]:
# Your code here

### B2 extended [Proposed]. Is the 2009 drawdown extreme?

**Task.** S&P 500 daily log returns 2000–2026: 5,000 bootstrap paths of the same length, iid and stationary (mean block 20 days); distribution of the maximum drawdown and of the longest spell under water; bootstrap p-values of the observed values; $\hat\rho_1$ and the variance ratio $VR(20)$.

**Interpretation.** Is −56.8% extreme for this return distribution? Which bootstrap gives the fatter drawdown tail, and why?

In [ ]:
# Your code here

### B3 [Solved]. Correlations and calendars

**Task.** Bitcoin vs S&P 500, 2022–2026: correlation over the whole period (i) on common days — join the prices on common days first, then take returns — (ii) with equities forward-filled on weekends, (iii) weekly; compare with the mean of the 252-day rolling correlation used in the lecture. Stock–bond (S&P 500 vs TLT) in 2010–2020 and 2022–2026, both estimators.

**Interpretation.** Why do the two estimators differ, and which one answers the question "what was the correlation in 2022–2026"?

In [ ]:
eq, btc, tlt = load('S&P 500'), load('Bitcoin'), load('US Treasuries 20y+ (TLT)')
cb = np.log(pd.concat([eq, btc], axis=1).dropna()).diff().dropna()
ffill = np.log(pd.concat([eq.reindex(btc.index).ffill(), btc], axis=1).loc['2021-12-31':]).diff().dropna().loc['2022':]
wp = pd.concat([eq, btc], axis=1).dropna().resample('W-FRI').last()   # common-day prices, Friday to Friday
weekly = np.log(wp[wp.index <= cb.index[-1]]).diff().dropna().loc['2022':]
roll_cb = cb.iloc[:, 0].rolling(252).corr(cb.iloc[:, 1])
print(f'BTC-S&P 2022-2026: common days {cb.loc["2022":].corr().iloc[0, 1]:.3f}, forward-filled {ffill.corr().iloc[0, 1]:.3f}, '
      f'weekly {weekly.corr().iloc[0, 1]:.3f} ({len(weekly)} weeks); mean 252-day rolling {roll_cb.loc["2022":].mean():.3f}')
sb = np.log(pd.concat([eq, tlt], axis=1).dropna()).diff().dropna(); roll_sb = sb.iloc[:, 0].rolling(252).corr(sb.iloc[:, 1])
for a, b in [('2010', '2020'), ('2022', '2026')]:
    print(f'stock-bond {a}-{b}: whole period {sb.loc[a:b].corr().iloc[0, 1]:.3f}, mean 252-day rolling {roll_sb.loc[a:b].mean():.3f}')

### B3 extended [Proposed]. Has the stock–bond correlation really changed?

**Task.** (a) Fisher z test (valid for iid bivariate-Normal pairs) for 2010–2020 vs 2022–2026. (b) Stationary bootstrap separately in each period (mean block 20 days): 95% interval for $\hat\rho_2 - \hat\rho_1$. (c) Compare the widths of the iid and bootstrap intervals. (d) Bitcoin–S&P 500 (2022–2026): intervals for the daily and the weekly correlation, and a joint bootstrap of their difference (weeks resampled in blocks of mean 4, both correlations recomputed from each resample).

**Interpretation.** Why is the bootstrap interval wider? Does the stock–bond change remain significant? And the daily–weekly difference of A7? Which economic event should explain a change of sign?

In [ ]:
# Your code here

### B4 [Proposed]. Stablecoin growth

**Task.** Stablecoin supply (DefiLlama): year-end values and growth; peak-to-trough decline in 2022–2023 with exact dates; compound annual growth since 1 Jan 2020.

**Interpretation.** Supply is a stock, not a price: what does a fall in supply mean economically, and would the same fall in a stablecoin's price mean the same?

In [ ]:
# Your code here

### B4 extended [Proposed]. Did the Terra collapse change the growth regime?

**Task.** $g_t = \Delta\ln(\text{supply}_t)$, daily, 2020–2026. (a) Known date (Chow, 1960): $g_t = a + b\,1\{t \ge \text{9 May 2022}\} + e_t$ with HAC standard errors (Newey and West, 1987); test $b = 0$. (b) Unknown date (Andrews, 1993): sup of the HAC Wald statistic with 15% trimming (5% critical value 8.85). (c) Least-squares break date and its 95% interval (Bai, 1997), which assumes a single break and the same error distribution in both regimes; check that assumption.

**Interpretation.** Did the slowdown start with Terra? Is the break in the level or in the drift of supply?

In [ ]:
# Your code here

### B5 [Proposed]. BVB in RON and in EUR

**Task.** BET-TR converted to euro with the BNR EUR/RON rate; CAGR and volatility since 5 Jan 2015 in both currencies; correlation of BET-TR returns with EUR/RON changes.

**Interpretation.** For a euro-based investor, is the leu's depreciation a cost or a hedge? What would change your answer?

In [ ]:
# Your code here

### B5 extended [Proposed]. Is the currency effect significant?

**Task.** Log returns $r^{EUR} = r^{RON} - \Delta s$, $s = \ln(\text{EUR/RON})$ (BNR). (a) Decompose $\operatorname{Var}(r^{EUR}) = \operatorname{Var}(r^{RON}) + \operatorname{Var}(\Delta s) - 2\operatorname{Cov}(r^{RON}, \Delta s)$, annualised. (b) Test Cov = 0: regression of $r^{RON}$ on $\Delta s$ with HAC standard errors; stationary-bootstrap interval for the correlation. (c) FX drag = mean of $\Delta s$ per year: 95% interval and HAC t statistic.

**Interpretation.** Is the 1.6 pp gap between the two CAGRs distinguishable from zero? Would a euro investor hedge the leu?

In [ ]:
# Your code here

### B6 [Solved]. Finding bad ticks

**Task.** EUR/RON series from EODHD (2015–2026): drop weekend rows and flag points more than 0.05 RON from an 11-day centred rolling median (an illustrative threshold); compare each flagged quote with the BNR rate of the same day and remove only confirmed errors (more than 0.05 RON from BNR); annualised volatility raw, cleaned and BNR.

**Interpretation.** Why does a gap remain between the cleaned series and the BNR rate after the errors are removed?

In [ ]:
m = load('EUR/RON (EODHD)', start='2015-01-01'); bnr = load('EUR/RON', start='2015-01-01')
vol = lambda x: np.log(x).diff().dropna().std() * np.sqrt(obs_per_year(x))   # FX: observed frequency, not 252
wd = m[m.index.dayofweek < 5]; flagged = wd.index.difference(clean_series(m, max_dev=0.05).index)
gap_bnr = pd.Series({d: abs(wd[d] - bnr.asof(d)) for d in flagged})                  # flagged quote vs BNR rate of the same day
confirmed = gap_bnr.index[gap_bnr > 0.05]; rule = wd.drop(confirmed)               # remove only confirmed errors
dev = (m - m.rolling(11, center=True, min_periods=3).median()).abs()
print(f'rows {len(m)}, weekend rows {int((m.index.dayofweek >= 5).sum())}, weekday points flagged {len(flagged)}, '
      f'confirmed errors removed {len(confirmed)} (distance to BNR {gap_bnr[confirmed].min():.3f}-{gap_bnr[confirmed].max():.3f} RON); '
      f'kept {[d.date().isoformat() for d in gap_bnr.index[gap_bnr <= 0.05]]} (within {gap_bnr[gap_bnr <= 0.05].max():.3f} RON of BNR)')
print(f'largest deviation {dev.max():.3f} RON on {dev.idxmax().date()}; vol raw {vol(m):.2%}, cleaned {vol(rule):.2%}, BNR {vol(bnr):.2%}')
j = np.log(pd.concat([rule, bnr], axis=1, keys=['mkt', 'bnr']).dropna()).diff().dropna()   # prices joined on common days first
ac = lambda x: np.log(x).diff().dropna().autocorr(1)                                        # each series on its own calendar
print(f"gap: corr of daily changes {j['mkt'].corr(j['bnr']):.2f}; lag-1 autocorrelation market {ac(rule):.2f}, BNR {ac(bnr):.2f}; "
      f'days quoted in the file but not fixed by BNR: {len(rule.index.difference(bnr.index))}')

### B6 extended [Proposed]. A calibrated threshold

**Task.** Hampel filter (Hampel, 1974): flag day $t$ if $|x_t - m_t| > c \cdot 1.4826 \cdot \text{MAD}_t$, with $m_t$ the median of the centred 11-day window and $\text{MAD}_t$ the median of $|x_j - m_t|$ over the same window (windows truncated at the ends of the sample). (a) On the BNR series 2015–2026, considered clean, report the share of flagged days for $c = 3, 5, 10, 20$ and find the smallest $c$ on the grid $1, 1.5, \ldots, 59.5$ that flags at most 0.1% of the days (an in-sample false-alarm rate); repeat with a floor on the threshold, $\max(c \cdot 1.4826 \cdot \text{MAD}_t, \text{floor})$, of 0.01, 0.02 and 0.03 RON (a modification of Hampel's rule). (b) Apply both calibrated rules to the EODHD series: how many days are flagged, and how many coincide with the 0.05 RON rule? (c) Volatility after cleaning vs BNR.

**Interpretation.** Why MAD rather than the standard deviation, and where does 1.4826 come from? In calm periods of the leu $\text{MAD}_t$ is almost zero: what happens to the filter, and which correction do you propose?

In [ ]:
# Your code here

### B7 [Solved]. Event study: the bank tax of December 2018

**Task.** The tax was announced on the evening of 18 December 2018; event day 0 = 19 December 2018. Banca Transilvania, daily log returns; BET as the market. (a) Market model $r_{i,t} = a_i + b_i r_{m,t} + \varepsilon_{i,t}$ on the window $[-250, -11]$ ($T = 240$ days). (b) Abnormal returns $AR_{i,t}$ and $CAR_i[0,+5]$. (c) $t$ tests with the prediction-error variances $\hat\sigma_\varepsilon^2[1 + 1/T + (r_{m,0} - \bar r_m)^2/S_{mm}]$ for $AR_0$ and $\hat\sigma_\varepsilon^2[L + L^2/T + (\sum r_{m} - L\bar r_m)^2/S_{mm}]$ for the CAR, $L = 6$, $S_{mm} = \sum (r_{m,t} - \bar r_m)^2$ over the estimation window; Student $t$ with $T - 2$ degrees of freedom.

**Interpretation.** BET contains both banks with large weights: what does this do to $\hat b_i$ and to the abnormal returns? Propose an alternative market index. The $t$ test assumes the variance does not change in the event window: is that plausible, and what would a cross-sectional test need? Was there a reversal in the following days, and what does it say about market efficiency (Chapter 2)?

In [ ]:
from scipy import stats
mkt = fetch_daily('BET', 'market', 'close', start='2017-01-01')
for name in ['Banca Transilvania']:
    r = np.log(pd.concat([load(name, start='2017-01-01'), mkt], axis=1, keys=['i', 'm']).dropna()).diff().dropna()
    t0 = r.index.get_loc(pd.Timestamp('2018-12-19')); est, ev = r.iloc[t0 - 250:t0 - 10], r.iloc[t0:t0 + 6]
    X = np.column_stack([np.ones(len(est)), est['m']]); b = np.linalg.lstsq(X, est['i'], rcond=None)[0]
    sig = (est['i'] - X @ b).std(ddof=2); ar = ev['i'] - (b[0] + b[1] * ev['m']); L, T = len(ar), len(est)
    # prediction-error variances, including the estimation error of a and b (MacKinlay, 1997)
    mb, Sxx = est['m'].mean(), ((est['m'] - est['m'].mean()) ** 2).sum()
    se0 = sig * np.sqrt(1 + 1 / T + (ev['m'].iloc[0] - mb) ** 2 / Sxx)
    seC = sig * np.sqrt(L + L ** 2 / T + (ev['m'].sum() - L * mb) ** 2 / Sxx)
    t0_, t = ar.iloc[0] / se0, ar.sum() / seC; pv = lambda z: 2 * (1 - stats.t.cdf(abs(z), T - 2))
    print(f'{name}: b = {b[1]:.2f}, sigma_e = {sig:.4f}; day 0: r_i {ev["i"].iloc[0]:.2%}, r_m {ev["m"].iloc[0]:.2%}, '
          f'AR_0 {ar.iloc[0]:.2%}, SE {se0:.2%} (sigma_e only: {sig:.2%}), t = {t0_:.1f}, p = {pv(t0_):.1e}; '
          f'CAR[0,+5] {ar.sum():.2%}, SE {seC:.2%} (sigma_e sqrt(L): {sig * np.sqrt(L):.2%}), t = {t:.2f}, p = {pv(t):.2f}')
    print('   AR by day:', [f'{v:+.2%}' for v in ar])

### B8 [Proposed]. The same event for BRD

**Task.** Repeat B7 for BRD (same window, market and test). (a) Report $\hat b$, $\hat\sigma_\varepsilon$, $AR_0$ with its $t$ and $CAR[0,+5]$ with $t$ and $p$. (b) Compare with Banca Transilvania: which bank was hit harder relative to its normal volatility? (c) The $t$ test assumes the variance does not change in the event window: explain the cross-sectional test of Boehmer, Musumeci and Poulsen (1991) and why it cannot simply be applied to one or two banks hit by the same event.

**Model:** B7 [Solved].

In [ ]:
# Your code here

## Part C: open question (project seed)

**C1. Did spot ETFs change Bitcoin's correlation with equities?** Spot Bitcoin ETFs were approved in the US on 10 January 2024. Did the Bitcoin–S&P 500 correlation rise *because of* them, or had it risen already since 2022? There is no single method: propose one and defend it (precise hypothesis; known-date test (Chow) vs unknown-date test (Andrews, 1993); robustness to the calendar, the window and the bootstrap block; a counterfactual with other assets). Deliverable: a Quantlet folder in your team repository and one slide with the conclusion and its main limitation.

In [ ]:
# Your code here

### C2 [Proposed]. Critique an AI answer

**Prompt** sent to an AI assistant: "Using the daily files in `data/market`, compute the annualised mean, the annualised volatility and the Sharpe ratio ($r_f = 0$) of SPY and Bitcoin from daily simple returns". **Answer:**

```python
spy, btc = load_file('SPY.US'), load_file('BTC-USD.CC')   # field='close'
A = 252
for name, p in [('SPY', spy), ('BTC', btc)]:
    r = p.pct_change().dropna(); mu = r.mean() * A; vol = r.std() * A
    print(name, mu, vol, mu / vol)
```

Conclusion of the AI: "Sharpe ratios: SPY 0.03, Bitcoin 0.07. Both assets barely reward their risk."

**Tasks.** (1) Find the three implementation errors. (2) Say how you would detect each. (3) Write the corrected code and report mean, volatility and Sharpe ratio (full files: SPY from 29 Jan 1993, Bitcoin from 13 Jul 2010). (4) State one limitation of comparing the two corrected Sharpe ratios. (5) Log every error you found in `AI_ERRORS.md`.

**Model:** A3–A4 and B1 [Solved].

In [ ]:
def load_file(sym, field='close'):
    """The full EODHD series (no start date)"""
    return read_market(sym)[field].dropna()

In [ ]:
# Your code here

### C3 [Proposed]. Passive investing and demand elasticity (research lab)

**Notation.** $w_{ik}$: shares of stock $k$ held by active investor $i$, as a fraction of all shares; $A_k = \sum_i w_{ik}$; $\mathcal{E}_{agg,k} = \sum_i w_{ik}\bar{\mathcal{E}}_{ik}/(1 + \chi A_k)$ (footnote 53 of Haddad, Huebner and Loualiche, 2025). **Tasks.** Two-investor warm-up; the ETF share from FRED (BOGZ1LM563064100Q / BOGZ1LM893064105Q); the simulation below for three switching rules, seeds 1–10 and $\chi \in \{2.97, 0\}$; compare each slope with the mean of $1/(1 + \chi A_k)$. The seed range is simulation variability, not a confidence interval.

**Model:** the lecture case study and B1 extended [Solved].

In [ ]:
def c3_run(seed, chi=2.97, K=2000, N=40, mode='rep'):
    """Simulated cross-section: K stocks, N active investors each; A_k = active share of stock k (fraction of all shares),
    w_ik = holdings of investor i in stock k as a fraction of all shares (sum_i w_ik = A_k),
    E_agg,k = sum_i w_ik Ebar_ik / (1 + chi A_k) (footnote 53); a fraction f_k of A_k turns passive under a rule."""
    rng = np.random.default_rng(seed)
    A = rng.uniform(0.55, 0.85, K); f = rng.uniform(0, 0.2, K)
    dlE, dlA, pt, E0s = np.empty(K), np.empty(K), np.empty(K), np.empty(K)
    for k in range(K):
        s = rng.dirichlet(np.ones(N)) * A[k]; e = np.exp(np.log(1.6) + 0.6 * rng.standard_normal(N))
        E0 = (s * e).sum() / (1 + chi * A[k]); target = f[k] * A[k]
        if mode == 'rep':
            sw = f[k] * s
        else:
            order = np.argsort(-e) if mode == 'high' else np.argsort(e)
            sw = np.zeros(N); rem = target
            for i in order:
                x = min(s[i], rem); sw[i] = x; rem -= x
                if rem <= 1e-15:
                    break
        A1 = A[k] - sw.sum(); E1 = ((s - sw) * e).sum() / (1 + chi * A1)
        dlE[k] = np.log(E1 / E0); dlA[k] = np.log(A1 / A[k]); pt[k] = 1 / (1 + chi * A[k]); E0s[k] = E0
    X = np.c_[np.ones(K), dlA]; b = np.linalg.lstsq(X, dlE, rcond=None)[0]
    return b[1], pt.mean(), E0s.mean()

In [ ]:
# Your code here